# Recurrent Neural Networks (RNN) Demonstration In Python

## Background

Historical stock market data along with technical indicators like RSI is used to capture price trends and momentum. The dataset is divided into **training data (2021 – Mid 2025)** and **testing data (Last 6 months of 2025)** to mimic real-world forecasting.

## Objective 
To build an LSTM-based model that learns temporal patterns in stock prices and predicts future values, assessing its effectiveness in time series forecasting.


### Note

Currently, only price, volume, and RSI are considered as input features. However, incorporating additional factors (like macroeconomic indicators, news sentiment, etc.) can further improve model accuracy.

### Import Libraries

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yfinance as yf

from sklearn.preprocessing import StandardScaler
from ta.momentum import RSIIndicator
import random


torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

### Import Data

In [2]:
train_df = pd.read_csv('nifty_train.csv')
test_df = pd.read_csv('nifty_test.csv')


In [3]:
train_df.head()

,Date,Price,Volume
0,2021-01-01,14018.50,258090905
1,2021-01-04,14132.90,494999295
2,2021-01-05,14199.50,492475349
3,2021-01-06,14146.25,632323316
4,2021-01-07,14137.35,559173512


In [4]:
test_df.head()

,Date,Price,Volume
0,2025-07-01,25541.8,260669106
1,2025-07-02,25453.4,309828013
2,2025-07-03,25405.3,293428797
3,2025-07-04,25461.0,193511595
4,2025-07-07,25461.3,196051345


### Calculate RSI
- RSI (Relative Strength Index) is a momentum indicator that measures the speed and magnitude of recent price changes to identify overbought or oversold conditions.

- It ranges from 0 to 100, where values above 70 indicate overbought and below 30 indicate oversold conditions.


In [5]:
train_df['RSI'] = RSIIndicator(close=train_df['Price'], window=14).rsi()
test_df['RSI']  = RSIIndicator(close=test_df['Price'], window=14).rsi()

# Drop NA created due to RSI window
train_df = train_df.dropna()
test_df  = test_df.dropna()

In [6]:
train_df

,Date,Price,Volume,RSI
13,2021-01-20,14644.70,623069963,70.485362
14,2021-01-21,14590.35,704646957,66.535232
15,2021-01-22,14371.90,776750347,53.546164
16,2021-01-25,14238.90,618631493,47.469996
17,2021-01-27,13967.50,660711117,37.995145
...,...,...,...,...
1108,2025-06-24,25044.35,450185468,56.706975
1109,2025-06-25,25244.75,260582584,60.941147
1110,2025-06-26,25549.00,428891818,66.325882
1111,2025-06-27,25637.80,563957748,67.724456


### Calculate Returns and Target

**Note**

we are not shifting the target explicitly now; instead, the future alignment is handled during sequence creation in the LSTM (by selecting y[i + seq_len] as the next step).

In [7]:
# Train
train_df['Return'] = train_df['Price'].pct_change() * 100
train_df['Target'] = (train_df['Return'] > 0).astype(int)
# Test
test_df['Return'] = test_df['Price'].pct_change() * 100
test_df['Target'] = (test_df['Return'] > 0).astype(int)

# Drop NA 
train_df = train_df.dropna()
test_df  = test_df.dropna()

In [8]:
train_df

,Date,Price,Volume,RSI,Return,Target
14,2021-01-21,14590.35,704646957,66.535232,-0.371124,0
15,2021-01-22,14371.90,776750347,53.546164,-1.497222,0
16,2021-01-25,14238.90,618631493,47.469996,-0.925417,0
17,2021-01-27,13967.50,660711117,37.995145,-1.906046,0
18,2021-01-28,13817.55,637871621,33.961801,-1.073564,0
...,...,...,...,...,...,...
1108,2025-06-24,25044.35,450185468,56.706975,0.290126,1
1109,2025-06-25,25244.75,260582584,60.941147,0.800180,1
1110,2025-06-26,25549.00,428891818,66.325882,1.205201,1
1111,2025-06-27,25637.80,563957748,67.724456,0.347567,1


 ### Features And Labels

In [9]:
features = ['Price', 'Volume', 'RSI']

X_train = train_df[features].values
y_train = train_df['Target'].values

X_test = test_df[features].values
y_test = test_df['Target'].values

### Scaling (Fit only on Train)


In [10]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)


### Create Sequence

- This function converts time series data into sequences so that the LSTM can learn temporal patterns.

- It uses the past seq_len days as input features and assigns the target as the value immediately after the sequence.

**Note (Understanding for i = 0, seq_len = 10)**

X[0:10]
- Takes values from index 0 to 9 (upper limit excluded)
- Corresponds to Day 1 to Day 10
- Includes all features: price, volume, RSI

y[10]
- Refers to the value at index 10
- This is the next value after the input sequence

“Using data from Day 1 to Day 10 → predict what happens next.”

In [11]:
seq_len = 10  # number of past time steps (days) to use

def create_sequences(X, y, seq_len):
    Xs, ys = [], []
    
    # Loop over the dataset to create sequences
    for i in range(len(X) - seq_len):
        
        # Take 'seq_len' consecutive rows as one input sequence
        Xs.append(X[i:i+seq_len])

        # Target is the value immediately AFTER the sequence
        ys.append(y[i+seq_len])
    
    # Convert lists to numpy arrays
    return np.array(Xs), np.array(ys)

# Apply sequence creation on train and test data
X_train, y_train = create_sequences(X_train, y_train, seq_len)
X_test, y_test   = create_sequences(X_test, y_test, seq_len)

### Convert To Pytorch Tensors

In [12]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

### Build LSTM Model

- An LSTM (Long Short-Term Memory) model is a type of recurrent neural network (RNN) designed to learn patterns from sequential data by remembering long-term dependencies.
- It is widely used in time series forecasting (like stock prices) because it captures both past trends and temporal relationships effectively.

In [13]:
# LSTM layer:
# input_size = 3 → number of features (Close, Volume, RSI)
# hidden_size = 8 → number of neurons in LSTM (model complexity)
# batch_first=True → input shape = (batch_size, seq_len, features)
lstm = nn.LSTM(input_size=3, hidden_size=8, batch_first=True)

# Fully connected layer:
# Takes LSTM output (size 8) and converts to single output (binary classification)
fc = nn.Linear(8, 1)

# Loss function:
# BCELoss → used for binary classification (output between 0 and 1)
loss_fn = nn.BCELoss()

# Optimizer:
# Adam optimizer updates model weights to minimize loss
optimizer = torch.optim.Adam(
    list(lstm.parameters()) + list(fc.parameters()),  # parameters to train
    lr=0.01  # learning rate
)

### Train the Model

- This loop trains the LSTM model over multiple epochs by performing forward pass, loss calculation, and backpropagation.
- The model predicts probabilities using sigmoid, compares them with actual targets using loss, and updates weights to improve performance.
- Training progress is monitored by printing loss at regular intervals.

In [14]:
# Training loop (runs for 20 iterations over the dataset)
for epoch in range(20):

    # -------------------------
    # Forward Pass
    # -------------------------
    out, _ = lstm(X_train_t)  
    # LSTM output shape: (batch_size, seq_len, hidden_size)
    
    out = fc(out[:, -1, :]).squeeze()  
    # Take last time step output and pass through FC layer

    pred = torch.sigmoid(out)  
    # Convert logits to probabilities (0 to 1)

    # -------------------------
    # Shape Adjustment
    # -------------------------
    pred = pred.view(-1)         
    y_train_t = y_train_t.view(-1)  
    # Ensure both have same shape for loss calculation

    # -------------------------
    # Loss Calculation
    # -------------------------
    loss = loss_fn(pred, y_train_t)  
    # Compare predicted vs actual values

    # -------------------------
    # Backpropagation
    # -------------------------
    optimizer.zero_grad()  # Clear previous gradients
    loss.backward()        # Compute gradients
    optimizer.step()       # Update weights

    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 5, Loss: 0.6877
Epoch 10, Loss: 0.6866
Epoch 15, Loss: 0.6854
Epoch 20, Loss: 0.6836


### Train Accuracy


In [15]:
train_pred = (pred > 0.5).int()
train_acc = (train_pred == y_train_t.int()).float().mean()

print("Train Accuracy:", train_acc.item())

Train Accuracy: 0.5592286586761475


### Test Accuracy

In [16]:
with torch.no_grad():
    out_test, _ = lstm(X_test_t)
    out_test = fc(out_test[:, -1, :]).squeeze()
    pred_test = torch.sigmoid(out_test)

test_pred = (pred_test > 0.5).int()
test_acc = (test_pred == y_test_t.int()).float().mean()

print("Test Accuracy:", test_acc.item())

Test Accuracy: 0.5098039507865906
